# 03 - EDA bivariado

Aquí analizaremos la relación entre cada predictor y la variable objetivo `cardio`.

Buscaremos:
- diferencias entre `cardio=0` y `cardio=1`,
- solapamiento de distribuciones,
- categorías con mayor proporción de `cardio=1`,
- asociaciones lineales exploratorias.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path("cardio_train.csv")
df = pd.read_csv(DATA_FILE, sep=";")

df["age_years"] = df["age"] / 365.25
df["bmi"] = df["weight"] / ((df["height"] / 100) ** 2)

df.head()

## 1. Variables numéricas vs `cardio`

In [ ]:
numeric_cols = ["age_years", "height", "weight", "ap_hi", "ap_lo", "bmi"]

for col in numeric_cols:
    print(f"\n--- {col} ---")
    display(
        df.groupby("cardio")[col]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .round(2)
    )

### Cómo interpretar

Compara especialmente:
- media,
- mediana,
- dispersión.

Si `cardio=1` presenta valores sistemáticamente mayores o menores, la variable puede contener señal predictiva.

## 2. Boxplots por `cardio`

In [ ]:
for col in numeric_cols:
    data0 = df.loc[df["cardio"] == 0, col].dropna()
    data1 = df.loc[df["cardio"] == 1, col].dropna()

    plt.figure(figsize=(8, 5))
    plt.boxplot([data0, data1], labels=["cardio=0", "cardio=1"])
    plt.title(f"{col} según cardio")
    plt.ylabel(col)
    plt.show()

## 3. Histogramas superpuestos

In [ ]:
for col in numeric_cols:
    data0 = df.loc[df["cardio"] == 0, col].dropna()
    data1 = df.loc[df["cardio"] == 1, col].dropna()

    plt.figure(figsize=(8, 5))
    plt.hist(data0, bins=40, alpha=0.5, label="cardio=0", density=True)
    plt.hist(data1, bins=40, alpha=0.5, label="cardio=1", density=True)
    plt.title(f"Distribución de {col} según cardio")
    plt.xlabel(col)
    plt.ylabel("Densidad")
    plt.legend()
    plt.show()

### Qué significa el solapamiento

Si ambas distribuciones se superponen casi completamente, la variable por sí sola separa poco las clases.

Si una distribución aparece claramente desplazada respecto a la otra, puede existir mayor capacidad discriminativa.

## 4. Variables categóricas vs `cardio`

In [ ]:
categorical_cols = ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]

for col in categorical_cols:
    print(f"\n--- {col} vs cardio ---")

    counts = pd.crosstab(df[col], df["cardio"])
    pct = pd.crosstab(df[col], df["cardio"], normalize="index") * 100

    print("Frecuencias:")
    display(counts)

    print("Porcentajes por categoría:")
    display(pct.round(2))

    pct.plot(kind="bar", stacked=True, figsize=(8, 5))
    plt.title(f"Proporción de cardio según {col}")
    plt.xlabel(col)
    plt.ylabel("Porcentaje")
    plt.legend(["cardio=0", "cardio=1"], title="cardio")
    plt.show()

## 5. Tasa de `cardio=1` por categoría

In [ ]:
for col in categorical_cols:
    rate = df.groupby(col)["cardio"].mean() * 100

    display(rate.round(2).to_frame("porcentaje_cardio_1"))

    plt.figure(figsize=(7, 4.5))
    plt.bar(rate.index.astype(str), rate.values)
    plt.title(f"Tasa de cardio=1 según {col}")
    plt.xlabel(col)
    plt.ylabel("Porcentaje con cardio=1")
    plt.show()

## 6. Correlación exploratoria con `cardio`

In [ ]:
corr_cols = [
    "age_years", "gender", "height", "weight",
    "ap_hi", "ap_lo", "cholesterol", "gluc",
    "smoke", "alco", "active", "bmi", "cardio"
]

corr_with_cardio = (
    df[corr_cols]
    .corr(numeric_only=True)["cardio"]
    .drop("cardio")
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

corr_with_cardio.round(3)

In [ ]:
plt.figure(figsize=(8, 6))
plt.barh(corr_with_cardio.index, corr_with_cardio.values)
plt.title("Correlación de cada variable con cardio")
plt.xlabel("Correlación de Pearson")
plt.show()

## 7. Conclusiones metodológicas

Recuerda:

- diferencia no implica causalidad;
- correlación baja no significa inutilidad predictiva;
- una variable puede aportar información solo al interactuar con otras;
- variables ordinales codificadas como 1/2/3 requieren cautela;
- los outliers pueden distorsionar los resultados.

Después de este EDA, el siguiente paso natural será **preprocesamiento y preparación de datos**.